# Paper figures walkthrough

## Goal

Inspect the six archived numerical tables, apply the public validation checks,
and reproduce every article figure with the same plotting functions used by
`scripts/plot_all.py`. The notebook contains no waveform implementation or
hidden calculation state.


## Setup

Locate the repository root, import the table, plotting, and validation helpers,
and load the single parameter source `configs/paper.yaml`.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "configs" / "paper.yaml").exists():
    root = root.parent
if not (root / "configs" / "paper.yaml").exists():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks/")

sys.path.insert(0, str(root / "src"))

from fsd_accelerating_waveforms.config import FIGURE_IDS, load_config
from fsd_accelerating_waveforms.plotting import plot_figure
from fsd_accelerating_waveforms.tables import read_rows
from fsd_accelerating_waveforms.validation import validate_generated_data

config = load_config(root / "configs" / "paper.yaml", mode="full")
data_dir = root / "data"
figure_dir = root / "build" / "notebook_figures"


## Steps

### 1. Inspect all six figure tables

The catalog below is derived from the configuration. Each table is read in the
same way and summarized by filename, row count, and field names.


In [ ]:
table_files = {
    "fig01": config["figures"]["fig01"]["data_file"],
    "fig02": config["mismatch"]["cases"]["fig02"]["data_file"],
    "fig03": config["mismatch"]["cases"]["fig03"]["data_file"],
    "fig04": config["mismatch"]["cases"]["fig04"]["data_file"],
    "fig05": config["fsd_order"]["data_file"],
    "fig06": config["fisher"]["data_file"],
}

table_inventory = []
for figure_id in FIGURE_IDS:
    path = data_dir / table_files[figure_id]
    rows = read_rows(path)
    table_inventory.append(
        {
            "figure": figure_id,
            "file": path.name,
            "rows": len(rows),
            "fields": tuple(rows[0]) if rows else (),
        }
    )

table_inventory


### 2. Replot all six figures

Outputs are written under ignored `build/notebook_figures/`. The archived
release figures and numerical tables are not modified.


In [ ]:
replotted = {
    figure_id: plot_figure(config, figure_id, data_dir, figure_dir)
    for figure_id in FIGURE_IDS
}
replotted


## Checks

Run the same artifact and scientific checks used by the command-line
reproduction workflow, then confirm that every configured figure was produced.


In [ ]:
validation = validate_generated_data(
    config,
    data_dir,
    figure_dir,
    require_all=True,
)
assert validation["status"] == "passed", validation["failures"]
assert set(replotted) == set(FIGURE_IDS)
assert all(path.exists() for path in replotted.values())
validation


## Next Steps

Use `python scripts/plot_all.py` for a direct table-to-figure run, or
`python scripts/reproduce_all.py --mode full` to recompute the tables before
plotting. Release provenance, dependency versions, runtime, memory use, and
hashes are recorded in `reproduction_record.json`.
